Since you are using Gemini API keys, please ensure your Gemini API key is set as an environment variable named `GOOGLE_API_KEY`.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-openai langchain-text-splitters \
               faiss-cpu tiktoken python-dotenv

In [ ]:
# Install the necessary library for Google Generative AI embeddings
!pip install -q langchain-google-genai

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings

#

## Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL

# Explicitly import YouTubeTranscriptApi and TranscriptsDisabled within this cell
# to ensure they are correctly recognized and available, mitigating potential environment issues.
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

# The diagnostic output showed that 'get_transcript' is not an attribute.
# The correct method for fetching a transcript is 'fetch'.
# Also, YouTubeTranscriptApi.fetch does not accept a 'languages' argument directly.
# We fetch all available transcripts and then select the English one.
try:
    # The persistent 'missing self' error suggests 'list' is an instance method.
    # Instantiate YouTubeTranscriptApi (even if unconventional for top-level list/fetch)
    api_instance = YouTubeTranscriptApi()
    transcript_list = api_instance.list(video_id=video_id)

    # Find the English transcript from the list
    english_transcript_object = None
    for transcript_object in transcript_list:
        if transcript_object.language_code == "en":
            english_transcript_object = transcript_object
            break

    if english_transcript_object:
        # Flatten it to plain text after fetching the specific transcript data
        # Corrected method from 'fetch_transcript()' to 'fetch()'
        transcript_data = english_transcript_object.fetch()
        # Corrected from chunk["text"] to chunk.text
        transcript = " ".join(chunk.text for chunk in transcript_data)
        print(transcript)
    else:
        print("English captions not available for this video.")

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:

transcript_list

## Step 1b - Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:

chunks[100]

# Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)


In [ ]:
# Initialize embeddings using the environment variable (Safe for GitHub)
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
print("✅ Embeddings model initialized successfully!")

In [ ]:
import time

# 1. Use a smaller batch size to stay well under the 100 RPM limit
batch_size = 20
vector_store = None

for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]

    try:
        if vector_store is None:
            # First batch creates the index
            vector_store = FAISS.from_documents(batch, embeddings)
        else:
            # Subsequent batches add to the index
            vector_store.add_documents(batch)

        print(f"✅ Successfully indexed chunks {i} to {i + len(batch)}")

        # 2. Wait 15-20 seconds. Even though the limit is 1 minute,
        # a short pause helps prevent 'burst' detection.
        time.sleep(20)

    except Exception as e:
        print(f"❌ Error at chunk {i}: {e}")
        print("Waiting 60 seconds to reset quota...")
        time.sleep(60) # If it fails, wait a full minute before trying that batch again

In [ ]:
vector_store.index_to_docstore_id


In [ ]:
vector_store.get_by_ids(['5f43bd5f-5c37-4675-acdd-7dbb3e42e0ea'])

#

## Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

In [ ]:
retriever.invoke('What is deepmind')

##Step-3 Augmentation

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)



In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,

    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:

context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

#

## Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

# Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')